# 04. Đánh giá và so sánh mô hình

Notebook đọc kết quả dự báo do `03_models.ipynb` tạo ra và thực hiện:

- Tính MAE, RMSE và sMAPE theo yêu cầu.
- Bổ sung MSE, R2 và các chỉ số phát hiện mưa.
- Xuất bảng `results/metrics.csv`.
- Vẽ đồ thị `y_true` và `y_pred`.
- Lưu hình vào thư mục `figures/`.

## 1. Thư viện và dữ liệu

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
)

plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = Path("../results")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = RESULTS_DIR / "three_models_predictions.csv"
CONFIG_PATH = RESULTS_DIR / "model_config.json"

if not PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        "Hãy chạy 03_models.ipynb trước để tạo file dự báo."
    )

predictions_df = pd.read_csv(
    PREDICTIONS_PATH,
    parse_dates=["target_time"],
)

if CONFIG_PATH.exists():
    with CONFIG_PATH.open("r", encoding="utf-8") as handle:
        CONFIG = json.load(handle)
else:
    CONFIG = {
        "rain_threshold_mm": 0.1,
        "prediction_threshold_mm": 0.1,
    }

display(predictions_df.head())

## 2. Tính các chỉ số đánh giá

In [ ]:
MODEL_COLUMNS = {
    "Linear Regression": "linear_regression_mm",
    "XGBoost": "xgboost_mm",
    "GRU": "gru_mm",
}

def smape(y_true, y_pred):
    denominator = (
        np.abs(y_true) + np.abs(y_pred)
    ) / 2.0
    ratio = np.divide(
        np.abs(y_true - y_pred),
        denominator,
        out=np.zeros_like(denominator, dtype=float),
        where=denominator > 1e-10,
    )
    return 100 * ratio.mean()

def evaluate_model(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    threshold = CONFIG["rain_threshold_mm"]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true >= threshold,
        y_pred >= threshold,
        average="binary",
        zero_division=0,
    )
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "sMAPE (%)": smape(y_true, y_pred),
        "MSE": mse,
        "R2": r2_score(y_true, y_pred),
        "Rain precision": precision,
        "Rain recall": recall,
        "Rain F1": f1,
    }

y_true = predictions_df["y_true_mm"].to_numpy()
rows = []

for model_name, column in MODEL_COLUMNS.items():
    rows.append(
        evaluate_model(
            model_name,
            y_true,
            predictions_df[column].to_numpy(),
        )
    )

comparison_df = (
    pd.DataFrame(rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
numeric_cols = comparison_df.select_dtypes(
    include=np.number
).columns
comparison_df[numeric_cols] = comparison_df[
    numeric_cols
].round(6)

comparison_df.to_csv(
    RESULTS_DIR / "metrics.csv",
    index=False,
)
display(comparison_df)

## 3. Đồ thị y_true và y_pred

In [ ]:
rain_indices = np.flatnonzero(
    y_true >= CONFIG["rain_threshold_mm"]
)
if len(rain_indices):
    center = int(rain_indices[0])
    left = max(0, center - 100)
    right = min(len(y_true), center + 250)
else:
    left = 0
    right = min(500, len(y_true))

plot_df = predictions_df.iloc[left:right]

plt.figure(figsize=(18, 6))
plt.plot(
    plot_df["target_time"],
    plot_df["y_true_mm"],
    label="Thực tế",
    color="#172B4D",
    linewidth=2.2,
    zorder=5,
)
plt.plot(
    plot_df["target_time"],
    plot_df["linear_regression_mm"],
    label="Linear Regression",
    color="#7A5195",
    linewidth=1.2,
)
plt.plot(
    plot_df["target_time"],
    plot_df["xgboost_mm"],
    label="XGBoost",
    color="#EF5675",
    linewidth=1.5,
)
plt.plot(
    plot_df["target_time"],
    plot_df["gru_mm"],
    label="GRU",
    color="#2A9D8F",
    linewidth=1.5,
)
plt.xlabel("Thời gian dự báo")
plt.ylabel("Lượng mưa (mm)")
plt.title("So sánh y_true và y_pred trên cùng khoảng test")
plt.legend(ncol=4)
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "y_true_vs_y_pred.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

## 4. Biểu đồ so sánh chỉ số

In [ ]:
ordered = comparison_df.set_index("Model").loc[
    ["Linear Regression", "XGBoost", "GRU"]
]
colors = ["#7A5195", "#EF5675", "#2A9D8F"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].bar(ordered.index, ordered["MAE"], color=colors)
axes[0].set_title("MAE")

axes[1].bar(ordered.index, ordered["RMSE"], color=colors)
axes[1].set_title("RMSE")

axes[2].bar(
    ordered.index,
    ordered["sMAPE (%)"],
    color=colors,
)
axes[2].set_title("sMAPE (%)")

for axis in axes:
    axis.tick_params(axis="x", rotation=20)
    axis.set_xlabel("")

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "model_metrics_comparison.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

## 5. Kết luận

In [ ]:
best_model = comparison_df.iloc[0]
print(
    "Mô hình có RMSE thấp nhất:",
    best_model["Model"],
)
print("MAE:", best_model["MAE"])
print("RMSE:", best_model["RMSE"])
print("sMAPE (%):", best_model["sMAPE (%)"])
print("\nĐã lưu:")
print("-", RESULTS_DIR / "metrics.csv")
print("-", FIGURES_DIR / "y_true_vs_y_pred.png")
print("-", FIGURES_DIR / "model_metrics_comparison.png")